# ZIP-to-BIDS

Convert selected DICOM acquisitions stored in ZIP archives on Google Drive into a BIDS dataset. Start with `INDEX_ONLY` on a new dataset, then run the full conversion once all expected Image IDs are found.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/sadeghghaderi93/zip-to-bids.git"
PROJECT_ROOT = Path("/content/zip-to-bids")

shutil.rmtree(PROJECT_ROOT, ignore_errors=True)
clone = subprocess.run(
    ["git", "clone", "-q", REPO_URL, str(PROJECT_ROOT)],
    capture_output=True,
    text=True,
)

if clone.returncode != 0:
    print("GitHub clone was not available. If the repository is private, upload its ZIP archive here.")
    from google.colab import files

    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No repository ZIP was uploaded.")
    archive = Path("/content") / next(iter(uploaded))
    extract_root = Path("/content/zip-to-bids-upload")
    shutil.rmtree(extract_root, ignore_errors=True)
    extract_root.mkdir(parents=True)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(extract_root)
    if (extract_root / "pyproject.toml").exists():
        PROJECT_ROOT.mkdir(parents=True)
        for item in extract_root.iterdir():
            shutil.move(str(item), str(PROJECT_ROOT / item.name))
    else:
        candidates = [p for p in extract_root.iterdir() if p.is_dir() and (p / "pyproject.toml").exists()]
        if not candidates:
            raise RuntimeError("The uploaded ZIP does not look like the ZIP-to-BIDS repository.")
        shutil.move(str(candidates[0]), str(PROJECT_ROOT))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)],
    check=True,
)

# Keep the source tree explicit as a fallback for Colab editable-install path quirks.
source_path = str(PROJECT_ROOT / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

import zip_to_bids
print(f"ZIP-to-BIDS {zip_to_bids.__version__} ready")


The next cell mounts Google Drive and opens the file pickers. dcm2niix is downloaded from the official latest stable Linux release when conversion starts.


In [ ]:
from zip_to_bids.colab import launch

launch()
